<a href="https://colab.research.google.com/github/Cado87/Monai_projects/blob/main/Spleen%20segmentation/Rust_NIfTI_visualizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Rust Toolchain
Run this cell to install `rustup` and the Rust compiler (`rustc`, `cargo`).

In [1]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!rustc --version

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-05-28 for version 1.96.0 (ac68faa20 2026-05-25)
info: downloading 6 components
        cargo downloading [               ]   10.64 MiB (0 B/s, ETA: 0s)
        cargo downloading [               ]   10.64 MiB (0 B/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (10.92 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.64 MiB (10.92 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.64 MiB (10.64 MiB/s, ETA: 1s)
        cargo downloading [#              ]   10.64 MiB (10.64 MiB/s, ETA: 1s)
       

### 2. Initialize the Rust Project
Let's create a new Cargo project called `nifti_viewer` and add the `nifti` crate as a dependency.

In [2]:
!cargo new nifti_viewer
%cd nifti_viewer
!cargo add nifti
!cargo add ndarray # Useful for multi-dimensional array manipulation

    Creating binary (application) `nifti_viewer` package
note: see more `Cargo.toml` keys and their definitions at https://doc.rust-lang.org/cargo/reference/manifest.html
/content/nifti_viewer
    Updating crates.io index
      Adding nifti v0.17.0 to dependencies
             Features:
             + ndarray
             + ndarray_volumes
             - nalgebra
             - nalgebra_affine
             - simba
    Updating crates.io index
     Locking 28 packages to latest Rust 1.96.0 compatible versions
    Updating crates.io index
      Adding ndarray v0.17.2 to dependencies
             Features:
             + std
             - approx
             - blas
             - matrixmultiply-threading
             - portable-atomic-critical-section
             - rayon
             - serde
    Updating crates.io index
     Locking 1 package to latest Rust 1.96.0 compatible version
      Adding ndarray v0.17.2


### 3. Write a Basic NIfTI Reader in Rust
This will write a simple Rust program to `src/main.rs` that attempts to load a NIfTI file and print its header information.

In [3]:
%%writefile src/main.rs
use nifti::{NiftiObject, ReaderOptions};

fn main() {
    // Replace this with the path to one of your U-Net output NIfTI files
    let file_path = "test.nii.gz";

    println!("Attempting to load NIfTI file: {}", file_path);

    // Read the NIfTI file
    match ReaderOptions::new().read_file(file_path) {
        Ok(obj) => {
            let header = obj.header();
            println!("Successfully loaded NIfTI file!");
            println!("Dimensions: {:?}", header.dim);
            println!("Data type: {:?}", header.datatype);
        },
        Err(e) => {
            println!("Error loading NIfTI file: {:?}. Please ensure you have downloaded a sample .nii.gz file to this directory.", e);
        }
    }
}

Overwriting src/main.rs


In [4]:
!cargo build

  Downloaded byteorder v1.5.0
  Downloaded approx v0.5.1
  Downloaded bytemuck v1.25.0
  Downloaded adler2 v2.0.1
  Downloaded autocfg v1.5.1
  Downloaded simd-adler32 v0.3.9
  Downloaded cfg-if v1.0.4
  Downloaded rawpointer v0.2.1
  Downloaded quote v1.0.45
  Downloaded num-integer v0.1.46
  Downloaded quick-error v2.0.1
  Downloaded num-derive v0.4.2
  Downloaded rgb v0.8.53
  Downloaded proc-macro2 v1.0.106
  Downloaded unicode-ident v1.0.24
  Downloaded num-traits v0.2.19
  Downloaded flate2 v1.1.9
  Downloaded nifti v0.17.0
  Downloaded matrixmultiply v0.3.10
  Downloaded crc32fast v1.5.0
  Downloaded byteordered v0.6.0
  Downloaded num-complex v0.4.6
  Downloaded either v1.16.0
  Downloaded miniz_oxide v0.8.9
  Downloaded ndarray v0.17.2
  Downloaded syn v2.0.117
  Downloaded ndarray v0.16.1
   Compiling autocfg v1.5.1
   Compiling proc-macro2 v1.0.106
   Compiling num-traits v0.2.19
   Compiling matrixmultiply v0.3.10
   Compiling unicode-ident v1.0.24
   Compiling quote v1.0.4

### 4. Download a Sample NIfTI file and Run
Let's get a small sample `.nii.gz` file so our Rust program can read it.

In [8]:
import nibabel as nib
import numpy as np

# Create a simple 3D numpy array to simulate a medical volume
data = np.zeros((32, 32, 32), dtype=np.int16)
data[10:22, 10:22, 10:22] = 1  # Add a dummy 'spleen' segment

# Create a NIfTI image using an identity affine matrix
img = nib.Nifti1Image(data, np.eye(4))

# Save the image to the current directory as test.nii.gz
filename = 'test.nii.gz'
nib.save(img, filename)

print(f"Dummy NIfTI file '{filename}' created successfully!")

Dummy NIfTI file 'test.nii.gz' created successfully!


In [9]:
!cargo run

    Finished `dev` profile [unoptimized + debuginfo] target(s) in 0.05s
     Running `target/debug/nifti_viewer`
Attempting to load NIfTI file: test.nii.gz
Successfully loaded NIfTI file!
Dimensions: [3, 32, 32, 32, 1, 1, 1, 1]
Data type: 4


### 5. Extract and Render a Slice using Rust
Let's add the `image` crate to process and save a 2D slice from our 3D volume as a PNG file.

In [10]:
!cargo add image

    Updating crates.io index
      Adding image v0.25.10 to dependencies
             Features:
             + avif
             + bmp
             + dds
             + default-formats
             + exr
             + ff
             + gif
             + hdr
             + ico
             + jpeg
             + png
             + pnm
             + qoi
             + rayon
             + tga
             + tiff
             + webp
             - avif-native
             - benchmarks
             - color_quant
             - nasm
             - serde
    Updating crates.io index
     Locking 90 packages to latest Rust 1.96.0 compatible versions
      Adding aligned v0.4.3
      Adding aligned-vec v0.6.4
      Adding anyhow v1.0.102
      Adding arbitrary v1.4.2
      Adding arg_enum_proc_macro v0.3.4
      Adding arrayvec v0.7.6
      Adding as-slice v0.2.1
      Adding av-scenechange v0.14.1
      Adding av1-grain v0.2.5
      Adding avif-serialize v0.8.9
      Adding bit_field v0.10.

In [14]:
%%writefile src/main.rs
use nifti::{IntoNdArray, NiftiObject, ReaderOptions};
use image::{ImageBuffer, Luma};

fn main() {
    let file_path = "test.nii.gz";
    println!("Loading NIfTI file: {}", file_path);

    // Read the NIfTI file and convert the volume to an ndarray
    let obj = ReaderOptions::new().read_file(file_path).expect("Failed to read NIfTI");
    let volume = obj.into_volume();
    let ndarray = volume.into_ndarray::<f32>().expect("Failed to convert to ndarray");

    let shape = ndarray.shape();
    println!("Data shape: {:?}", shape);

    // We know our dummy data is 3D. Let's get the middle slice on the Z axis.
    let width = shape[0];
    let height = shape[1];
    let depth = shape[2];
    let z_slice = depth / 2;

    // Create a new grayscale image buffer
    let mut imgbuf = ImageBuffer::new(width as u32, height as u32);

    for x in 0..width {
        for y in 0..height {
            // Access the voxel value
            let val = ndarray[[x, y, z_slice]];

            // Normalize the value. Ensure it's an 8-bit unsigned integer (u8)
            // Our dummy data has values 0 (background) and 1 (spleen segment).
            let pixel_val: u8 = if val > 0.5 { 255 } else { 0 };

            imgbuf.put_pixel(x as u32, y as u32, Luma([pixel_val]));
        }
    }

    // Save the resulting slice as a PNG
    let output_file = "slice.png";
    imgbuf.save(output_file).expect("Failed to save image");
    println!("Successfully saved middle slice to {}", output_file);
}

Overwriting src/main.rs


In [12]:
!cargo run

  Downloaded arrayvec v0.7.6
  Downloaded aligned v0.4.3
  Downloaded aligned-vec v0.6.4
  Downloaded arg_enum_proc_macro v0.3.4
  Downloaded anyhow v1.0.102
  Downloaded byteorder-lite v0.1.0
  Downloaded zune-core v0.5.1
  Downloaded paste v1.0.15
  Downloaded equator-macro v0.4.2
  Downloaded profiling v1.0.18
  Downloaded maybe-rayon v0.1.1
  Downloaded avif-serialize v0.8.9
  Downloaded equator v0.4.2
  Downloaded as-slice v0.2.1
  Downloaded wasm-bindgen-shared v0.2.125
  Downloaded y4m v0.8.0
  Downloaded wasm-bindgen-macro v0.2.125
  Downloaded stable_deref_trait v1.2.1
  Downloaded new_debug_unreachable v1.0.6
  Downloaded pastey v0.1.1
  Downloaded loop9 v0.1.5
  Downloaded thiserror-impl v2.0.18
  Downloaded v_frame v0.3.9
  Downloaded zune-inflate v0.2.54
  Downloaded log v0.4.32
  Downloaded weezl v0.1.12
  Downloaded zune-jpeg v0.5.15
  Downloaded half v2.7.1
  Downloaded zerocopy-derive v0.8.52
  Downloaded png v0.18.1
  Downloaded nom v8.0.0
  Downloaded moxcms v0.8.1
 

In [13]:
from IPython.display import Image, display
import os

if os.path.exists('slice.png'):
    print("Displaying the rendered NIfTI slice:")
    display(Image('slice.png', width=200))
else:
    print("slice.png not found. Make sure the Rust code ran successfully.")

slice.png not found. Make sure the Rust code ran successfully.
